# configspace

## Example 1: a space from a dict, sampled reproducibly

In [ ]:
# Example 1: tuples become integer/float ranges, lists become categoricals; a seed makes sampling repeatable
from ConfigSpace import ConfigurationSpace

cs = ConfigurationSpace({"layers": (1, 8), "lr": (1e-4, 1e-1), "act": ["relu", "tanh"]}, seed=1)
a = cs.sample_configuration(5)
b = ConfigurationSpace({"layers": (1, 8), "lr": (1e-4, 1e-1), "act": ["relu", "tanh"]}, seed=1).sample_configuration(5)
assert [dict(c) for c in a] == [dict(c) for c in b]
assert all(1 <= c["layers"] <= 8 and c["act"] in ("relu", "tanh") for c in a)
print("Example 1:", dict(a[0]))

## Example 2: typed hyperparameters with a log scale and a default

In [ ]:
# Example 2: Integer/Float/Categorical with explicit defaults; the default configuration uses them
from ConfigSpace import Categorical, ConfigurationSpace, Float, Integer

cs = ConfigurationSpace(seed=0)
cs.add([Integer("batch", (16, 512), log=True, default=64), Float("lr", (1e-5, 1e-1), log=True, default=1e-3),
        Categorical("opt", ["adam", "sgd"], default="adam")])
default = cs.get_default_configuration()
assert dict(default) == {"batch": 64, "lr": 1e-3, "opt": "adam"}, dict(default)
assert cs["lr"].log and cs["batch"].log
print("Example 2:", dict(default))

## Example 3: a conditional hyperparameter is active only for one parent value

In [ ]:
# Example 3: momentum exists only when opt == "sgd"; sampled configurations respect the condition
from ConfigSpace import Categorical, ConfigurationSpace, EqualsCondition, Float

cs = ConfigurationSpace(seed=3)
opt, momentum = Categorical("opt", ["adam", "sgd"]), Float("momentum", (0.0, 0.99))
cs.add([opt, momentum, EqualsCondition(momentum, opt, "sgd")])
for config in cs.sample_configuration(20):
    assert ("momentum" in config) == (config["opt"] == "sgd"), dict(config)
print("Example 3:", [dict(c) for c in cs.sample_configuration(2)])

## Example 4: a forbidden combination is never sampled

In [ ]:
# Example 4: ForbiddenAndConjunction rules out (kernel=linear, degree=5)
from ConfigSpace import Categorical, ConfigurationSpace, ForbiddenAndConjunction, ForbiddenEqualsClause, Integer

cs = ConfigurationSpace(seed=7)
kernel, degree = Categorical("kernel", ["linear", "poly"]), Integer("degree", (1, 5))
cs.add([kernel, degree, ForbiddenAndConjunction(ForbiddenEqualsClause(kernel, "linear"), ForbiddenEqualsClause(degree, 5))])
samples = cs.sample_configuration(200)
assert not any(c["kernel"] == "linear" and c["degree"] == 5 for c in samples)
print("Example 4:", len(samples), "samples, none with kernel=linear and degree=5")

## Example 5: write the space to JSON and read it back

In [ ]:
# Example 5: to_json/from_json round-trip a space with a condition; neighbours of a configuration stay valid
import tempfile, pathlib
from ConfigSpace import Categorical, ConfigurationSpace, EqualsCondition, Float
from ConfigSpace.util import get_one_exchange_neighbourhood

cs = ConfigurationSpace(seed=5)
opt, momentum = Categorical("opt", ["adam", "sgd"]), Float("momentum", (0.0, 0.99))
cs.add([opt, momentum, EqualsCondition(momentum, opt, "sgd")])
path = pathlib.Path(tempfile.mkdtemp()) / "space.json"
cs.to_json(path)
assert ConfigurationSpace.from_json(path) == cs
neighbours = list(get_one_exchange_neighbourhood(cs.sample_configuration(), seed=1))
assert neighbours and all(n.config_space == cs for n in neighbours)
print("Example 5: JSON round trip equal;", len(neighbours), "neighbours")